# Labs 8-9 - Ensemble methods


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from collections import Counter
from scipy.stats import chi2
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None


## Task 1 - own AdaBoost implementation


In [ ]:
class AdaBoostTrees(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=100, max_depth=1, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        if len(self.classes_) != 2:
            raise ValueError("This implementation is binary only.")
        y_signed = np.where(y == self.classes_[1], 1, -1)
        n = len(y)
        w = np.ones(n) / n
        self.estimators_, self.alphas_, self.errors_ = [], [], []
        for k in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=self.max_depth, random_state=self.random_state + k)
            stump.fit(X, y, sample_weight=w)
            pred = stump.predict(X)
            incorrect = pred != y
            err = float(np.sum(w * incorrect))
            if err <= 1e-12:
                self.estimators_.append(stump)
                self.alphas_.append(20.0)
                self.errors_.append(err)
                break
            if err >= 0.5:
                # A weak learner no better than random voting would get non-positive weight.
                break
            beta = err / (1 - err)
            alpha = np.log(1 / beta)
            w[~incorrect] *= beta
            w /= w.sum()
            self.estimators_.append(stump)
            self.alphas_.append(alpha)
            self.errors_.append(err)
        return self

    def decision_function(self, X):
        score = np.zeros(X.shape[0])
        for est, alpha in zip(self.estimators_, self.alphas_):
            pred = np.where(est.predict(X) == self.classes_[1], 1, -1)
            score += alpha * pred
        return score

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, self.classes_[1], self.classes_[0])


In [ ]:
X_real, y_real = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X_real, y_real, test_size=0.3, stratify=y_real, random_state=RANDOM_STATE)
for depth in [1, 2, 4]:
    clf = AdaBoostTrees(n_estimators=100, max_depth=depth, random_state=RANDOM_STATE).fit(X_train, y_train)
    print(depth, len(clf.estimators_), accuracy_score(y_test, clf.predict(X_test)))

for err in [0.4, 1e-8]:
    print("error", err, "vote weight log((1-error)/error) =", np.log((1 - err) / err))


Smaller weighted error gives a larger vote weight. A near-perfect classifier can dominate because its weight is very large. Stumps often work well because boosting converts many weak low-variance rules into a strong additive classifier.


## Task 2 - compare ensemble methods


In [ ]:
def make_chi_square_data(n, random_state=42):
    local_rng = np.random.default_rng(random_state)
    X = local_rng.normal(size=(n, 10))
    cutoff = chi2.ppf(0.5, df=10)
    y = np.where(np.sum(X ** 2, axis=1) > cutoff, 1, 0)
    return X, y


X_art_train, y_art_train = make_chi_square_data(2000, RANDOM_STATE)
X_art_test, y_art_test = make_chi_square_data(10000, RANDOM_STATE + 1)
print("Artificial class balance:", y_art_train.mean())


The artificial dataset is balanced because the threshold is the median of a chi-square(10) distribution. The Bayes boundary is a sphere in 10 dimensions, so flexible tree ensembles should approximate it better than a shallow single tree.


In [ ]:
def build_methods():
    methods = {
        "single_tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "bagging": BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=RANDOM_STATE),
        "adaboost_own": AdaBoostTrees(n_estimators=100, max_depth=1, random_state=RANDOM_STATE),
        "gradient_boosting": GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    }
    if XGBClassifier is not None:
        methods["xgboost"] = XGBClassifier(n_estimators=100, eval_metric="logloss", random_state=RANDOM_STATE)
    else:
        methods["xgboost_fallback"] = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, random_state=RANDOM_STATE)
    return methods


def compare_methods(X_train, y_train, X_test, y_test):
    rows = []
    for name, clf in build_methods().items():
        clf.fit(X_train, y_train)
        rows.append({
            "method": name,
            "train_error": 1 - accuracy_score(y_train, clf.predict(X_train)),
            "test_error": 1 - accuracy_score(y_test, clf.predict(X_test)),
        })
    return rows


real_split = train_test_split(X_real, y_real, test_size=0.3, stratify=y_real, random_state=RANDOM_STATE)
print("real data")
compare_methods(real_split[0], real_split[2], real_split[1], real_split[3])


In [ ]:
print("artificial data")
compare_methods(X_art_train, y_art_train, X_art_test, y_art_test)


In [ ]:
def staged_errors(model_factory, X_train, y_train, X_test, y_test, grid=(10, 25, 50, 100, 150)):
    rows = []
    for n_estimators in grid:
        model = model_factory(n_estimators)
        model.fit(X_train, y_train)
        rows.append((n_estimators, 1 - accuracy_score(y_train, model.predict(X_train)), 1 - accuracy_score(y_test, model.predict(X_test))))
    return np.asarray(rows)


curves = {
    "bagging": staged_errors(lambda n: BaggingClassifier(DecisionTreeClassifier(), n_estimators=n, random_state=RANDOM_STATE), X_art_train, y_art_train, X_art_test, y_art_test),
    "random_forest": staged_errors(lambda n: RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE), X_art_train, y_art_train, X_art_test, y_art_test),
    "gradient_boosting": staged_errors(lambda n: GradientBoostingClassifier(n_estimators=n, random_state=RANDOM_STATE), X_art_train, y_art_train, X_art_test, y_art_test),
    "adaboost": staged_errors(lambda n: AdaBoostTrees(n_estimators=n, max_depth=1, random_state=RANDOM_STATE), X_art_train, y_art_train, X_art_test, y_art_test),
}
for name, arr in curves.items():
    plt.plot(arr[:, 0], arr[:, 2], marker="o", label=name)
plt.xlabel("iterations / trees")
plt.ylabel("test error")
plt.grid(True)
plt.legend()
plt.show()


Bagging and random forest mainly reduce variance. Boosting and gradient boosting often reduce bias by sequentially focusing on hard cases, with some variance control through shrinkage and shallow trees. Random forest can outperform bagging because feature subsampling reduces tree correlation before averaging.
